# Covariance Shrinkage (Ledoit–Wolf)

06/04 showed that a sample covariance matrix of many assets from few observations has eigenvalues that are too spread out,
and 07/03–07/04 fixed it by imposing a factor structure. Shrinkage is the simplest general fix: take a weighted average of the
sample covariance matrix and a structured target,

$$
\hat\Sigma_\delta = \delta F + (1 - \delta)S
$$

Ledoit and Wolf showed how to choose $\delta$ from the data. This notebook derives the optimal intensity as a bias-variance
trade-off, implements the Ledoit–Wolf estimators for two targets, and measures what shrinkage does for minimum-variance portfolios.

## 0. Math link

- [Regularization](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/05_point_estimation/05_regularization.ipynb):
  accept a little bias to reduce variance; shrinkage is regularization of a covariance matrix.
- [Sample variance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/04_sampling_and_point_estimation/04_sample_variance.ipynb):
  the sample covariance matrix is unbiased but noisy.
- [Spectral decomposition](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/055_spectral_decomposition.ipynb):
  shrinkage toward a multiple of the identity moves every eigenvalue toward their average and keeps the eigenvectors.
- Noisy eigenvalues and minimum-variance portfolios from [06/04](../06_estimation_error/04_means_are_hard_covariances_are_easier.ipynb),
  factor-model covariance from [07/03](../07_regression_and_factor_models/03_multi_factor_models_and_return_attribution.ipynb),
  ridge regression from [07/02](../07_regression_and_factor_models/02_least_squares_and_the_normal_equations.ipynb).

## 1. Introduction

A portfolio manager estimates the covariance matrix of 50 stocks from five years of monthly returns and builds the
minimum-variance portfolio. In sample it looks very safe. Its true volatility is more than twice the optimum (07/03). Shrinking the
sample matrix halfway toward a simple target fixes most of the problem, and the right amount of shrinkage can be estimated from
the same data.

After this notebook you can:

- write the shrinkage estimator and choose a target (scaled identity, constant correlation);
- derive the optimal shrinkage intensity as a bias-variance trade-off;
- implement the Ledoit–Wolf intensity estimators;
- explain what shrinkage does to eigenvalues and condition numbers;
- compare sample, shrunk and factor-model covariance matrices on minimum-variance portfolios.

## 2. Theory

### The estimator

Let $S$ be the sample covariance matrix (unbiased, high variance) and $F$ a **target** (biased, low variance). For $\delta \in [0, 1]$:

$$
\hat\Sigma_\delta = \delta F + (1-\delta)S
$$

Common targets:

- **scaled identity** $F = \bar\sigma^2 I$ with $\bar\sigma^2 = \operatorname{tr}(S)/N$: all variances equal, no correlation (Ledoit and Wolf 2004);
- **constant correlation** $F_{ii} = s_{ii}$, $F_{ij} = \bar r\sqrt{s_{ii}s_{jj}}$ with $\bar r$ the average sample correlation: keeps each asset's variance, replaces
  every correlation by the average (Ledoit and Wolf 2003);
- **factor model** $F = \hat B^\top\hat\Sigma_f\hat B + D$ (07/03).

### The optimal intensity

Measure the error with the Frobenius norm $\lVert A\rVert^2 = \sum_{ij}a_{ij}^2$. If $E[S] = \Sigma$ and, as a first approximation, $F$ is non-random:

$$
E\lVert\hat\Sigma_\delta - \Sigma\rVert^2 = \delta^2\lVert F - \Sigma\rVert^2 + (1-\delta)^2 E\lVert S - \Sigma\rVert^2
$$

(the cross term vanishes because $E[S - \Sigma] = 0$). Setting the derivative in $\delta$ to zero:

$$
\delta^* = \frac{E\lVert S - \Sigma\rVert^2}{E\lVert S - \Sigma\rVert^2 + \lVert F - \Sigma\rVert^2}
$$

Shrink more when the sample matrix is noisy (few observations, many assets) and less when the target is far from the truth.
The minimized error is $\delta^*\lVert F - \Sigma\rVert^2$, always below $E\lVert S - \Sigma\rVert^2$.

### Estimating $\delta^*$

$\Sigma$ is unknown, so both terms must be estimated. With centred observations $x_t$ (rows of the data, $t = 1..n$):

- the **noise** $E\lVert S - \Sigma\rVert^2$ is estimated from how much the single-period cross-products $x_tx_t^\top$ scatter around $S$:
  $\hat b^2 = \frac{1}{n^2}\sum_t\lVert x_tx_t^\top - S\rVert^2$;
- the **total distance** $\lVert F - \Sigma\rVert^2 + E\lVert S - \Sigma\rVert^2$ is estimated by $\hat d^2 = \lVert S - F\rVert^2$.

For the identity target, $\hat\delta = \min(\hat b^2, \hat d^2)/\hat d^2$ (Ledoit and Wolf 2004). For the constant-correlation target the formula has extra
terms because $F$ itself is estimated; the Python section implements it.

### What shrinkage does to eigenvalues

With $F = \bar\sigma^2 I$ and $S = V\Lambda V^\top$:

$$
\hat\Sigma_\delta = V\big(\delta\bar\sigma^2 I + (1-\delta)\Lambda\big)V^\top
$$

The eigenvectors are unchanged; each eigenvalue moves linearly toward the average $\bar\sigma^2$. Small eigenvalues (06/04: too small) rise,
large ones (too large) fall, and the condition number drops. The matrix is positive definite for any $\delta > 0$, even when $N > n$.

### Connection to ridge regression

Minimum-variance weights solve $\hat\Sigma w = c\mathbf 1$. With the identity target, $\hat\Sigma_\delta = (1-\delta)(S + \frac{\delta}{1-\delta}\bar\sigma^2 I)$: the same $+\lambda I$ as ridge
regression (07/02, exercise 7). Both are regularization, and both can be read as a Bayesian prior (MAP) pulling the estimate
toward the target.

## 3. Worked Examples

### Example 1: a 2 × 2 matrix

$S = \begin{pmatrix}0.04 & 0.03\\ 0.03 & 0.04\end{pmatrix}$, identity target $\bar\sigma^2 = 0.04$, $\delta = 0.3$:

$$
\hat\Sigma = 0.3\begin{pmatrix}0.04 & 0\\ 0 & 0.04\end{pmatrix} + 0.7\begin{pmatrix}0.04 & 0.03\\ 0.03 & 0.04\end{pmatrix} = \begin{pmatrix}0.04 & 0.021\\ 0.021 & 0.04\end{pmatrix}
$$

Eigenvalues of $S$: 0.07 and 0.01 (condition number 7). Shrunk: $0.3\cdot 0.04 + 0.7\cdot 0.07 = 0.061$ and $0.3\cdot 0.04 + 0.7\cdot 0.01 = 0.019$ (condition
number 3.2). The correlation falls from 0.75 to 0.525.

### Example 2: the trade-off

Suppose $E\lVert S - \Sigma\rVert^2 = 0.5$ and $\lVert F - \Sigma\rVert^2 = 1.5$ (arbitrary units). Then $\delta^* = 0.5/2.0 = 0.25$, and the error is
$0.25^2\cdot 1.5 + 0.75^2\cdot 0.5 = 0.094 + 0.281 = 0.375$, 25% below the sample's 0.5.

### Example 3: constant-correlation target

Swiss equities (16%) and Swiss bonds (5%) in a universe with average correlation 0.3: the target covariance is
$0.3\cdot 0.16\cdot 0.05 = 0.0024$, whatever their sample correlation. Their variances stay at their sample values.

## 4. Finance Application

### Where it matters

- **Minimum-variance and risk-parity portfolios** use only the covariance matrix; shrinkage is the standard way to make it usable
  for 50–500 assets.
- **Risk systems** report shrinkage intensities; a high intensity is a warning that the sample matrix is mostly noise for that
  universe and window.
- **Tracking-error optimization** against a benchmark inverts the covariance matrix just like the minimum-variance problem.
- **Short histories.** New funds, crypto assets, or a switch to weekly data after a regime change: shrinkage lets you use the
  sample information without trusting it fully.

### The usual misreadings

- "Shrinkage throws away information." It reweights it: the sample matrix is still used, with a weight chosen to minimize
  expected error.
- "Shrinkage intensity 0.7 means correlations are 70% wrong." It means that, given the sample size, the estimated noise is large
  relative to the distance to the target.
- "Shrink toward the identity for any universe." The identity target ignores that equities are correlated with each other; for a
  stock universe the constant-correlation or factor target is closer to the truth, so the data support shrinking much further
  toward it (a smaller $\lVert F - \Sigma\rVert^2$ raises $\delta^*$) and the result is better.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

factors = ["MKT", "SMB", "HML", "MOM"]
f_mu = np.array([0.06, 0.02, 0.03, 0.05])
f_vol = np.array([0.16, 0.08, 0.09, 0.12])
f_corr = np.array([[1.00, 0.20, -0.10, -0.15],
                   [0.20, 1.00, 0.05, 0.00],
                   [-0.10, 0.05, 1.00, -0.30],
                   [-0.15, 0.00, -0.30, 1.00]])
Sigma_f = np.outer(f_vol, f_vol) * f_corr

N = 50
B_true = np.column_stack([1 + 0.25 * rng.standard_normal(N), 0.2 + 0.4 * rng.standard_normal(N),
                          0.1 + 0.4 * rng.standard_normal(N), 0.2 * rng.standard_normal(N)]).T
idio = rng.uniform(0.15, 0.35, N)                     # idiosyncratic vols differ across stocks
Sigma_true = B_true.T @ Sigma_f @ B_true + np.diag(idio**2)   # annual covariance
L_true = np.linalg.cholesky(Sigma_true / 12)
mu_m = (B_true.T @ f_mu) / 12

def simulate(n_obs, rng):
    return mu_m + rng.standard_normal((n_obs, N)) @ L_true.T

### Ledoit–Wolf estimators

Two targets: scaled identity (Ledoit–Wolf 2004) and constant correlation (Ledoit–Wolf 2003). Inputs are monthly returns; outputs
are annualized covariance matrices.

In [ ]:
def lw_identity(X, ppy=12):
    n, p = X.shape
    Xc = X - X.mean(axis=0)
    S = Xc.T @ Xc / n                                      # MLE scaling (1/n), as in Ledoit-Wolf
    m = np.trace(S) / p
    F = m * np.eye(p)
    d2 = np.sum((S - F) ** 2)
    b2_bar = np.sum([np.sum((np.outer(x, x) - S) ** 2) for x in Xc]) / n**2
    b2 = min(b2_bar, d2)
    delta = b2 / d2
    return (delta * F + (1 - delta) * S) * ppy, delta


def lw_constant_corr(X, ppy=12):
    n, p = X.shape
    Xc = X - X.mean(axis=0)
    S = Xc.T @ Xc / n
    sd = np.sqrt(np.diag(S))
    R = S / np.outer(sd, sd)
    r_bar = (R.sum() - p) / (p * (p - 1))
    F = r_bar * np.outer(sd, sd)
    np.fill_diagonal(F, np.diag(S))
    Y = Xc[:, :, None] * Xc[:, None, :]                     # (n, p, p) cross-products
    pi_mat = ((Y - S) ** 2).mean(axis=0)
    pi_hat = pi_mat.sum()
    Z = Xc**2 - np.diag(S)                                  # (n, p)
    theta_ii = np.einsum("ti,tij->ij", Z, Y - S) / n        # Cov-type term for (s_ii, s_ij)
    rho_off = (r_bar / 2) * ((np.outer(1 / sd, sd)) * theta_ii + (np.outer(sd, 1 / sd)) * theta_ii.T)
    np.fill_diagonal(rho_off, 0.0)
    rho_hat = np.trace(pi_mat) + rho_off.sum()
    gamma_hat = np.sum((F - S) ** 2)
    kappa = (pi_hat - rho_hat) / gamma_hat
    delta = max(0.0, min(1.0, kappa / n))
    return (delta * F + (1 - delta) * S) * ppy, delta


X60 = simulate(60, rng)
S60 = np.cov(X60, rowvar=False) * 12
Sig_id, d_id = lw_identity(X60)
Sig_cc, d_cc = lw_constant_corr(X60)
print(f"shrinkage intensity: identity target {d_id:.3f}, constant-correlation target {d_cc:.3f}")

try:
    from sklearn.covariance import LedoitWolf
    print(f"sklearn LedoitWolf intensity (identity target): {LedoitWolf().fit(X60).shrinkage_:.3f}")
except ImportError:
    print("sklearn not installed; skipping the cross-check")

Our implementation of the identity-target intensity matches scikit-learn (0.145). The constant-correlation target gets a much
higher intensity, 0.68: it is much closer to the true covariance of a stock universe, so the formula trusts it more.

### Eigenvalues and condition numbers

In [ ]:
def eig_desc(A):
    return np.sort(np.linalg.eigvalsh(A))[::-1]

ev = pd.DataFrame({"true": eig_desc(Sigma_true), "sample": eig_desc(S60),
                   "LW identity": eig_desc(Sig_id), "LW const corr": eig_desc(Sig_cc)})
print(ev.iloc[[0, 1, 2, 9, 24, 39, 49]].round(4).to_string())
print("\ncondition numbers:", {c: f"{ev[c].iloc[0] / ev[c].iloc[-1]:.0f}" for c in ev.columns})

lam_s, V_s = np.linalg.eigh(np.cov(X60, rowvar=False, bias=True))
lam_i, V_i = np.linalg.eigh(Sig_id / 12)
print("identity shrinkage keeps eigenvectors:", np.allclose(np.abs(V_s), np.abs(V_i)))

The sample matrix's smallest eigenvalue is 0.0011 against a true 0.0238, and its condition number is 1,594 against 62. The
identity target raises the small eigenvalues (condition number 98); the constant-correlation target gets closest to the true
spectrum (67), because it also fixes the largest eigenvalues.

### Which estimator gives the best minimum-variance portfolio?

200 repetitions for each sample length; true volatility of the minimum-variance portfolio built from each estimate, and the
Frobenius error of each estimate.

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(S.shape[0]))
    return w / w.sum()

def true_vol(w):
    return np.sqrt(w @ Sigma_true @ w)

gmv_opt = true_vol(gmv(Sigma_true))
results = []
for n_obs in [24, 60, 120, 240]:
    for rep in range(200):
        X = simulate(n_obs, rng)
        S = np.cov(X, rowvar=False) * 12
        ests = {"sample": S if n_obs > N else S + 1e-10 * np.eye(N),
                "LW identity": lw_identity(X)[0], "LW const corr": lw_constant_corr(X)[0]}
        for name, E in ests.items():
            results.append({"n": n_obs, "estimator": name, "GMV true vol": true_vol(gmv(E)),
                            "Frobenius error": np.linalg.norm(E - Sigma_true)})
results = pd.DataFrame(results)
table = results.groupby(["n", "estimator"]).mean().unstack("estimator")
print(f"true optimum: {gmv_opt:.2%}")
table["GMV true vol"].style.format("{:.2%}")

In [ ]:
table["Frobenius error"].round(3)

With 60 months the minimum-variance portfolio from the sample matrix has a true volatility of 30.4%, against an optimum of 12.5%; both
shrinkage estimators bring it to about 15.5%. With 240 months the sample matrix is nearly as good (14.0% against 13.7%). With 24
months the sample matrix is singular; the code adds $10^{-10}I$ to invert it, a tiny ridge that makes this case look better than
$n = 60$, but neither is usable. The constant-correlation estimate has the smallest Frobenius error at every sample size.

### Oracle intensity vs estimated intensity

For $n = 60$, compute the error for a grid of $\delta$ with the identity target in each repetition, and compare the best fixed $\delta$ with the
Ledoit–Wolf estimate.

In [ ]:
deltas = np.linspace(0, 1, 41)
err_grid = np.zeros(deltas.size)
vol_grid = np.zeros(deltas.size)
lw_deltas = []
for rep in range(200):
    X = simulate(60, rng)
    Xc = X - X.mean(axis=0)
    S = Xc.T @ Xc / 60 * 12
    F = np.trace(S) / N * np.eye(N)
    for i, d in enumerate(deltas):
        E = d * F + (1 - d) * S
        err_grid[i] += np.linalg.norm(E - Sigma_true) ** 2 / 200
        vol_grid[i] += true_vol(gmv(E + 1e-12 * np.eye(N))) / 200
    lw_deltas.append(lw_identity(X)[1])
print(f"delta minimizing Frobenius error: {deltas[np.argmin(err_grid)]:.3f}; "
      f"delta minimizing GMV true vol: {deltas[np.argmin(vol_grid)]:.3f}; "
      f"average Ledoit-Wolf delta: {np.mean(lw_deltas):.3f}")

The Ledoit–Wolf intensity (0.19 on average) is close to the intensity that minimizes the Frobenius error (0.15). The intensity that
minimizes the true volatility of the minimum-variance portfolio is higher (0.43): the Frobenius norm weights all entries equally,
while the optimizer is most sensitive to the smallest eigenvalues, which benefit from stronger shrinkage.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
k = np.arange(1, N + 1)
for col, st in [("true", "k-"), ("sample", "o-"), ("LW identity", "s-"), ("LW const corr", "^-")]:
    ax.semilogy(k, ev[col], st, markersize=3, label=col)
ax.set_title("Eigenvalues of the 50 × 50 covariance matrix, estimated from 60 months")
ax.set_xlabel("rank")
ax.set_ylabel("eigenvalue (annual variance, log scale)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
gv = table["GMV true vol"]
ax.plot(gv.index[1:], gv["sample"].iloc[1:], "o-", label="sample (singular at 24 months, not shown)")
for col, mk in [("LW identity", "s-"), ("LW const corr", "^-")]:
    ax.plot(gv.index, gv[col], mk, label=col)
ax.axhline(gmv_opt, color="black", linewidth=1, label="true optimum")
ax.set_xscale("log")
ax.set_xticks(gv.index)
ax.set_xticklabels(gv.index)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.set_title("True volatility of the minimum-variance portfolio (50 stocks, average of 200)")
ax.set_xlabel("months of data")
ax.set_ylabel("true annual volatility")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(deltas, err_grid / err_grid[0], color="tab:blue", label="Frobenius error² (relative to sample)")
ax1.plot(deltas, vol_grid / gmv_opt, color="tab:orange", label="GMV true vol / optimum")
ax1.axvline(np.mean(lw_deltas), color="black", linestyle="--", label=f"average Ledoit-Wolf delta {np.mean(lw_deltas):.2f}")
ax1.set_title("Identity-target shrinkage, 60 months: error vs intensity")
ax1.set_xlabel("shrinkage intensity delta")
ax1.set_ylabel("relative value")
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.show()

## 7. Exercises

### Basic

1. Shrink $S$ of Example 1 with $\delta = 0.5$ toward the identity target. Compute the eigenvalues, condition number and implied correlation.
2. What is $\delta^*$ when the sample noise $E\lVert S - \Sigma\rVert^2$ equals the target's bias $\lVert F - \Sigma\rVert^2$? By how much does the error fall?
3. A risk system reports a shrinkage intensity of 0.7 for the covariance matrix of 300 stocks estimated from two years of weekly
   data. **Interpret** this number and say what you would do with the minimum-variance portfolio built from the raw sample matrix.

### Intermediate

4. Show numerically that for the constant-correlation target the shrunk matrix keeps every sample variance and replaces each
   correlation by a weighted average of the sample correlation and $\bar r$.
5. With $n = 24$ months (fewer than the 50 assets) the sample matrix is singular. Show that both shrunk matrices are positive definite and
   compute their condition numbers.

### Advanced

6. Derive $\delta^*$ by differentiating $\delta^2 a + (1-\delta)^2 b$, and show the minimized value is $ab/(a+b) = \delta^* a$.
7. Compare linear shrinkage with the eigenvalue clipping of 07/04 (exercise 7) and with a four-component PCA factor model on
   minimum-variance portfolios at $n = 60$.

## 8. Solutions / Checks

In [ ]:
# 1.
S_ex = np.array([[0.04, 0.03], [0.03, 0.04]])
E1 = 0.5 * 0.04 * np.eye(2) + 0.5 * S_ex
l1 = np.linalg.eigvalsh(E1)
print(f"1. eigenvalues {l1}, condition {l1[1] / l1[0]:.2f}, correlation {E1[0, 1] / 0.04:.3f}")

# 2.
print("2. delta* = 0.5, error = a/2: half of the sample error")

# 4.
Xc = X60 - X60.mean(axis=0)
S_b = Xc.T @ Xc / 60 * 12
sd_b = np.sqrt(np.diag(S_b))
R_s = S_b / np.outer(sd_b, sd_b)
r_bar = (R_s.sum() - N) / (N * (N - 1))
R_shr = Sig_cc / np.outer(np.sqrt(np.diag(Sig_cc)), np.sqrt(np.diag(Sig_cc)))
print(f"4. variances kept: {np.allclose(np.diag(Sig_cc), np.diag(S_b))};  "
      f"correlations = delta r_bar + (1 - delta) r: {np.allclose(R_shr, d_cc * r_bar + (1 - d_cc) * R_s + d_cc * (1 - r_bar) * np.eye(N))}")

# 5.
X24 = simulate(24, rng)
S24 = np.cov(X24, rowvar=False) * 12
for name, E in [("sample", S24), ("LW identity", lw_identity(X24)[0]), ("LW const corr", lw_constant_corr(X24)[0])]:
    lmin, lmax = np.linalg.eigvalsh(E)[[0, -1]]
    cond = f"{lmax / lmin:.1f}" if lmin > 1e-12 else f"singular (rank {np.linalg.matrix_rank(E)})"
    print(f"5. {name:13s}: smallest eigenvalue {lmin:.2e}, condition {cond}")

In [ ]:
# 7.
def clipped_cov(S, n_obs):
    dd = np.sqrt(np.diag(S))
    C = S / np.outer(dd, dd)
    l, v = np.linalg.eigh(C)
    lp = (1 + np.sqrt(S.shape[0] / n_obs)) ** 2
    l_new = l.copy()
    l_new[l < lp] = l[l < lp].mean()
    C_new = v @ np.diag(l_new) @ v.T
    dn = np.sqrt(np.diag(C_new))
    return C_new / np.outer(dn, dn) * np.outer(dd, dd)

def pca_cov(X):
    # statistical factor model with 4 components (07/04)
    S = np.cov(X, rowvar=False) * 12
    l, v = np.linalg.eigh(S)
    l, v = l[::-1], v[:, ::-1]
    SK = v[:, :4] @ np.diag(l[:4]) @ v[:, :4].T
    return SK + np.diag(np.diag(S - SK))

r7 = {"sample": [], "LW identity": [], "LW const corr": [], "clipping": [], "PCA K=4": []}
for rep in range(200):
    X = simulate(60, rng)
    S = np.cov(X, rowvar=False) * 12
    r7["sample"].append(true_vol(gmv(S)))
    r7["LW identity"].append(true_vol(gmv(lw_identity(X)[0])))
    r7["LW const corr"].append(true_vol(gmv(lw_constant_corr(X)[0])))
    r7["clipping"].append(true_vol(gmv(clipped_cov(S, 60))))
    r7["PCA K=4"].append(true_vol(gmv(pca_cov(X))))
print("7.", {k: f"{np.mean(v):.2%}" for k, v in r7.items()}, f"optimum {gmv_opt:.2%}")

**1.** $\hat\Sigma = \begin{pmatrix}0.04 & 0.015\\ 0.015 & 0.04\end{pmatrix}$, eigenvalues 0.055 and 0.025, condition number 2.2, correlation 0.375.

**2.** $\delta^* = a/(a + a) = 0.5$, and the error falls to $a/2$: half of the sample matrix's error.

**3.** With 300 stocks and about 104 weekly observations the sample matrix is singular ($N > n$) and its eigenvalues are mostly noise; an
intensity of 0.7 says the estimated noise is large relative to how far the sample is from the target. A minimum-variance portfolio
from the raw sample matrix is meaningless (the matrix cannot even be inverted) or extremely concentrated in noise directions; use
the shrunk matrix or a factor model, and treat in-sample risk numbers with suspicion.

**4.** The diagonal is untouched, and each off-diagonal correlation becomes $\delta\bar r + (1-\delta)r_{ij}$.

**5.** The sample matrix has rank 23 (24 observations minus one for the mean), so it is singular. The shrunk matrices have smallest
eigenvalues of 0.034 and 0.024 and condition numbers of 32 and 65.

**6.** $\frac{d}{d\delta}[\delta^2 a + (1-\delta)^2 b] = 2\delta a - 2(1-\delta)b = 0 \Rightarrow \delta^* = b/(a+b)$ with $a = \lVert F - \Sigma\rVert^2$, $b = E\lVert S - \Sigma\rVert^2$. Substituting gives
$ab^2/(a+b)^2 + a^2b/(a+b)^2 = ab/(a+b) = \delta^* a$.

**7.** All four structured estimators beat the raw sample matrix (30.6%) by a wide margin. Eigenvalue clipping (14.7%) and the PCA
factor model (14.7%) do best here, ahead of linear shrinkage toward constant correlation (15.4%) and the identity (15.8%); the
optimum is 12.5%. Linear shrinkage moves all eigenvalues by the same fraction, while clipping and PCA treat signal and noise
eigenvalues differently, which suits a universe with a few strong factors.

## 9. Conclusion

- Shrinkage $\delta F + (1-\delta)S$ trades the bias of a simple target against the variance of the sample matrix.
- The optimal intensity is $\delta^* = E\lVert S-\Sigma\rVert^2/(E\lVert S-\Sigma\rVert^2 + \lVert F - \Sigma\rVert^2)$; Ledoit and Wolf estimate both terms from the data.
- Identity-target shrinkage keeps the eigenvectors and pulls the eigenvalues linearly toward their mean; the result is well
  conditioned even when $N > n$.
- For a stock universe the constant-correlation target is closer to the truth than the identity; both cut the true volatility of
  minimum-variance portfolios far below what the sample matrix delivers.
- Shrinkage is regularization, like ridge regression, and can be read as a prior centred on the target.

Next: [08/03 Risk parity](./03_risk_parity.ipynb).